# Chapter 01-02 · Essential Python for data work

**Label:** Optional  |  **Time:** ~45 minutes  |  **Difficulty:** gentle

**Prerequisites:** 01-01 (the diagnostic). If you passed tasks 1-3 there, skim this chapter and
read only the failure lab - it catches people who have written Python for years.

**Position in the learning path:** module 01, chapter 2 of 6. Before: **01-01**. After:
**01-03** (NumPy).

---

## Why this matters

Data work uses a surprisingly small slice of Python. Perhaps a dozen constructs carry almost
everything, and being fluent in those twelve is worth more than knowing forty you use once a
year.

But two of them hide a trap that produces **wrong numbers rather than error messages** - the
worst kind of bug, and one that shows up in ML code specifically, because ML code is full of
functions that accumulate results across many runs. The failure lab is the real content of this
chapter.

## What you will be able to do

By the end of this chapter you can:

1. **Choose** between a list and a dictionary for a given piece of data, and say why.
2. **Write** comprehensions that filter and transform, and know when a plain loop is better.
3. **Write** functions with sensible defaults, returning more than one value.
4. **Use** `zip`, `enumerate`, `sorted(key=...)` and `min/max(key=...)` fluently.
5. **Diagnose** the two aliasing bugs that make Python quietly return results from a previous
   run.

## Warm-up: retrieve, do not reread

From memory:

1. What is a confounder?
2. Why can a variable be an excellent predictor and a useless lever?
3. In the diagnostic, why was "how many rows does this join return?" the real question?
4. What does `axis=0` collapse?

<br>

*Answers: (1) something that influences both who gets the treatment and the outcome, so a plain
comparison mixes the two. (2) it can carry information about something else - emailing everyone
destroys the information without changing the cause. (3) because a join can silently multiply
rows, duplicating observations across a train/test split. (4) the axis you name is the one that
disappears - `axis=0` collapses the rows, leaving one value per column.*

## The situation

A colleague sends you the sensor log from the bike stand as a text dump, because the database
export is broken and it is Friday. Before pandas is involved at all, it is a list of records, and
you need to answer three questions: which sensors are running warm, what the average is per
location, and which reading is the worst.

This is not an artificial exercise. Data arrives as lists of dictionaries constantly - from JSON
APIs, from log files, from `csv.DictReader`, from database drivers - and the moment before you
call `pd.DataFrame(...)` is a real moment where plain Python is the right tool.

**The question this chapter answers:** which handful of Python constructs actually carry data
work, and where do they betray you?

In [ ]:
readings = [
    {"sensor": "s1", "site": "north", "temp_c": 18.5, "ok": True},
    {"sensor": "s2", "site": "north", "temp_c": 24.1, "ok": True},
    {"sensor": "s3", "site": "south", "temp_c": 31.7, "ok": False},
    {"sensor": "s4", "site": "south", "temp_c": 22.0, "ok": True},
    {"sensor": "s5", "site": "east", "temp_c": 27.3, "ok": True},
    {"sensor": "s6", "site": "east", "temp_c": 19.4, "ok": False},
]

print(f"{len(readings)} records, each a dict with keys: {list(readings[0])}")
print("first record:", readings[0])

### Lists and dictionaries, and which to reach for

- A **list** is an ordered sequence. Reach for it when position or order means something, and
  when you will iterate over everything.
- A **dictionary** maps keys to values. Reach for it when you will look things up **by name**,
  and when each item has several named parts.

A list of dictionaries - which is what arrived - says "many records, each with named fields". It
is the natural shape for row-oriented data, and it is exactly what a DataFrame is built from.

Two facts about dictionaries that matter later:

- Lookup by key is fast no matter how big the dictionary is. Scanning a *list* to find a matching
  record is not. If you find yourself looping over one list to match items in another, you want a
  dictionary - or a join (01-05).
- Since Python 3.7 dictionaries keep insertion order, so iterating over one is reproducible.
  That matters more than it sounds: reproducibility is a whole chapter later (04-08).

In [ ]:
# Indexing and slicing. Slices exclude the end, always.
print("first          :", readings[0]["sensor"])
print("last           :", readings[-1]["sensor"])
print("first three    :", [r["sensor"] for r in readings[:3]])
print("every second   :", [r["sensor"] for r in readings[::2]])

record = readings[2]
print("\nlookup by key  :", record["site"])
print("safe lookup    :", record.get("humidity", "not recorded"))

`readings[:3]` gives elements 0, 1 and 2 - **the end is excluded**. This is consistent everywhere
in Python and NumPy, and it has one genuinely useful consequence: `a[:k]` and `a[k:]` split a
sequence with no overlap and no gap, which is exactly what you want for a train/test split.

`record["humidity"]` would raise `KeyError`. `record.get("humidity", default)` returns the
default instead. Use `[...]` when a missing key means something is wrong and you want to know
immediately; use `.get(...)` when absence is expected and has a sensible substitute. Choosing
deliberately between "fail loudly" and "carry on quietly" is a habit worth building now - it is
the same decision as filling in a missing value, which chapter 02-04 spends its length on.

### Predict before running

1. What does `readings[1:1]` return - an error, `None`, or something else?
2. `sites = [r["site"] for r in readings]` - how many items, and are duplicates removed?
3. `sorted(readings, key=lambda r: r["temp_c"])[0]` - what is this expression *for*?

In [ ]:
print("empty slice        :", readings[1:1])
sites = [r["site"] for r in readings]
print("sites              :", sites)
print("unique sites       :", sorted(set(sites)))
print("coolest reading    :", min(readings, key=lambda r: r["temp_c"])["sensor"])

An empty slice returns an **empty list**, not an error. Convenient, and occasionally the reason a
loop silently does nothing.

`set(sites)` removes duplicates; `sorted(...)` around it makes the order deterministic, because a
set has no order and printing one directly can vary. Whenever you print or iterate a set in code
that someone else will run, sort it - unstable output is a small thing that wastes a large amount
of time.

`min(readings, key=...)` is the idiom worth stealing: **`key` says what to compare by, and the
whole item comes back.** `min(temps)` gives you a number; `min(readings, key=...)` gives you the
record, so you can ask which sensor it was. The same applies to `max` and `sorted`, and you will
use it constantly in error analysis - "show me the ten rows with the largest error" is exactly
this.

In [ ]:
# Comprehensions: filter, transform, or both.
warm = [r["sensor"] for r in readings if r["temp_c"] > 25]
in_f = [round(r["temp_c"] * 9 / 5 + 32, 1) for r in readings]
flags = {r["sensor"]: r["ok"] for r in readings}          # a dict comprehension

print("warm sensors   :", warm)
print("in fahrenheit  :", in_f)
print("flag lookup    :", flags["s3"])

Read a comprehension in three parts, right to left after the first:

```
[ r["sensor"]      for r in readings      if r["temp_c"] > 25 ]
  what I want       where it comes from    which ones
```

The dictionary version uses `{key: value for ...}` and builds a lookup table in one line - here,
sensor name to status flag, so you can check any sensor instantly instead of scanning the list.

**When not to use one.** If a comprehension needs two conditions, a nested loop and a conditional
expression, write the loop. Comprehensions are for making a simple transformation obvious, not for
proving that it fits on one line. The test is whether a colleague can read it once.

In [ ]:
def summarise(values, decimals=2):
    """Return (count, mean, spread) for a list of numbers. Empty input gives zeros and None."""
    if not values:
        return 0, None, None
    mean = sum(values) / len(values)
    spread = max(values) - min(values)
    return len(values), round(mean, decimals), round(spread, decimals)


count, mean, spread = summarise([r["temp_c"] for r in readings])
print(f"{count} readings, mean {mean} °C, range {spread} °C")
print("empty input ->", summarise([]))

Three things in that function are worth copying:

- **A default argument** (`decimals=2`) makes the common call short and the unusual call possible.
- **Returning a tuple** and unpacking it into three names is how Python returns several values.
  `train, test = split(...)` is the same move, and you will see it constantly.
- **The guard comes first.** Handling the empty case at the top keeps the main path unindented and
  readable.

One rule that pays for itself: **a function should do one thing and return a value, not print.**
`summarise` computes; the caller decides how to display it. A function that prints cannot be
tested, reused inside another calculation, or called in a loop without producing noise - and
almost every helper in this course ends up being called in a loop.

In [ ]:
# zip and enumerate: the two loop idioms you will use most.
sensors = [r["sensor"] for r in readings]
temps = [r["temp_c"] for r in readings]

for position, (sensor, temp) in enumerate(zip(sensors, temps), start=1):
    if temp > 25:
        print(f"{position}. {sensor} is warm at {temp} °C")

print("\npaired into a dict:", dict(zip(sensors, temps)))

`zip` walks several sequences together; `enumerate` adds a counter. Both avoid the `range(len(x))`
pattern, which is where off-by-one errors live.

**The `zip` trap worth knowing now:** if the sequences have different lengths, `zip` stops at the
shortest and says nothing. Two lists that were supposed to line up, one of which lost a row in an
earlier filter, will silently produce a shorter result. If they must be the same length,
`zip(a, b, strict=True)` raises instead - and in data work they almost always must be.

---

## Failure lab: results from a run you already finished

Here is a helper of the kind everybody writes when comparing models. It records a result and
returns the log so far.

**Predict before running:** we call it three times for experiment A, then start a fresh
experiment B and call it once more. How many entries will experiment B's log contain?

In [ ]:
def record(score, log=[]):          # <- looks harmless
    log.append(score)
    return log


experiment_a = record(0.81)
experiment_a = record(0.84)
experiment_a = record(0.79)
print("experiment A log:", experiment_a)

experiment_b = record(0.62)         # a brand new experiment
print("experiment B log:", experiment_b)

### Diagnosis

Experiment B's log contains **four** scores, three of which belong to experiment A.

**Why.** A default argument is evaluated **once**, when the function is defined - not each time it
is called. So there is exactly one list, created when Python read the `def` line, and every call
that does not pass a `log` appends to that same list. `experiment_a` and `experiment_b` are not
two lists; they are two names for one list.

**Why this is the worst kind of bug.** Nothing raises. The numbers are real numbers, the code
looks obviously correct, and the mean of experiment B is quietly contaminated by an earlier run.
In a notebook it is worse still, because re-running a cell keeps the accumulated state - which is
why "restart the kernel and run all" is the first thing to try when results stop making sense.

### The fix

In [ ]:
def record_safely(score, log=None):
    log = [] if log is None else log     # a fresh list per call, unless one was supplied
    log.append(score)
    return log


a = record_safely(0.81); a = record_safely(0.84, a)
b = record_safely(0.62)
print("A:", a, " B:", b)

`None` is the sentinel meaning "nothing was supplied", and the fresh list is created **inside**
the call, where it belongs.

**The rule, memorised as a rule:** *never use a mutable object - list, dict, set - as a default
argument value.* Numbers, strings, `True`, `False` and `None` are immutable and perfectly safe.

### The same bug, wearing different clothes

In [ ]:
baseline_config = {"model": "linear", "features": ["temp_c"]}

tuned_config = baseline_config                   # NOT a copy - a second name for the same dict
tuned_config["features"].append("site")
tuned_config["model"] = "tree"

print("tuned   :", tuned_config)
print("baseline:", baseline_config, "   <- changed too")

`tuned_config = baseline_config` copies the *reference*, not the object. There is one dictionary
with two names, and the "baseline" you intended to compare against no longer exists.

This is the single most common way an ML experiment ends up comparing a configuration against
itself. The fixes, in order of how much they copy:

| Approach | Copies | Use when |
|---|---|---|
| `dict(original)` or `original.copy()` | the top level only | The values are numbers and strings |
| `copy.deepcopy(original)` | everything, recursively | The values are lists or nested dicts, as here |
| Build a new dict: `{**original, "model": "tree"}` | top level, and states the change | Usually the clearest - the diff is visible in the code |

The nesting is what catches people: `baseline_config.copy()` would still share the *same*
`features` list, so `.append("site")` would still leak. Test it in the exercises.

In [ ]:
import copy

baseline = {"model": "linear", "features": ["temp_c"]}
shallow = baseline.copy()
deep = copy.deepcopy(baseline)

shallow["features"].append("site")
print("after touching the shallow copy, baseline is:", baseline)

baseline = {"model": "linear", "features": ["temp_c"]}
deep = copy.deepcopy(baseline)
deep["features"].append("site")
print("after touching the deep copy,   baseline is:", baseline)

The rule to carry: **assignment never copies in Python.** It binds a name. If you need an
independent object, say so explicitly.

This is not a Python quirk you can leave behind - the same distinction returns in pandas as views
versus copies (01-04, and the `SettingWithCopyWarning` that everyone meets and few read), and in
NumPy where a slice of an array *shares memory* with the original.

## Common misconceptions

**"`0.1 + 0.2 == 0.3` is true."**
It is `False`. Floating-point numbers are binary approximations of decimals, and the tiny error
is real. Never compare floats with `==`; use `abs(a - b) < 1e-9`, or `math.isclose`, or NumPy's
`np.allclose` for arrays. This bites in tests that compare a computed metric against an expected
one.

**"`is` and `==` mean the same thing."**
`==` asks whether two values are equal; `is` asks whether they are the *same object*. Small
integers and short strings are often cached, so `is` appears to work and then stops working on
larger values. Use `is` only for `None`, `True` and `False`.

**"An empty list is a fine default for 'nothing yet'."**
It is - as a local variable. As a *default argument* it is the failure lab above.

**"`5 / 2` is 2."**
`/` always gives a float (`2.5`); `//` gives the floor (`2`). Getting this wrong in an index
calculation gives a `TypeError`, which is the good case; getting it wrong in a rate calculation
gives a wrong number, which is not.

**"Mutating a list while looping over it is fine if I am careful."**
It is not. Removing items shifts the positions of everything after them, and the loop skips
elements. Build a new list with a comprehension instead - which is one of the reasons
comprehensions are the idiom.

**"Python is slow, so I should write clever code."**
Python *loops* are slow. The answer is not clever loops, it is not looping - which is the whole
subject of the next chapter. Write the obvious loop first, and reach for NumPy when a measurement
says you need to.

---

## Exercises

Solutions: `solutions/01_python_bridge/01-02_python_essentials_solutions.ipynb`.

### Quick understanding

**E1 (define).** When would you choose a dictionary over a list? Give one example from this
chapter's data where each is the right shape.

**E2 (explain).** Explain in two sentences why `def record(score, log=[])` returns results from
previous calls.

**E3 (explain).** What is the difference between `record["site"]` and `record.get("site")`, and
when do you want each?

### Hand calculation

**E4 (calculate).** Without running anything, write down what each expression gives for
`xs = [10, 20, 30, 40, 50]`: (a) `xs[1:3]`, (b) `xs[:-1]`, (c) `xs[::-1]`, (d) `xs[3:1]`,
(e) `len(xs[2:])`.

**E5 (calculate).** `a = [1, 2, 3]`, `b = a`, `c = a[:]`. Then `b.append(4)`. Write down what
`a`, `b` and `c` each contain, and say why.

### Coding

**E6 (code).** Write `mean_by_site(readings)` returning a dict mapping each site to the mean
`temp_c` at that site. Use only plain Python. Then write it a second way, using a different
construct from the first.

**E7 (code).** Write `worst_n(readings, n=2)` returning the `n` records with the highest
`temp_c`, hottest first. Use `sorted` with a `key`.

### Interpretation

**E8 (interpret).** `sum(r["ok"] for r in readings)` returns `4`. Explain why adding up booleans
works, and give one situation where relying on that would be a bad idea.

### Debugging

**E9 (diagnose).** This is meant to drop the broken sensors. Run it on the list below, by hand
first, then in code. Say what it returns, why, and give two different fixes.

```python
def drop_broken(records):
    for r in records:
        if not r["ok"]:
            records.remove(r)
    return records

batch = [{"id": 1, "ok": True}, {"id": 2, "ok": False},
         {"id": 3, "ok": False}, {"id": 4, "ok": True}]
```

Then run it on this chapter's `readings` instead, where it happens to give the right answer, and
say why that is worse news rather than better.

### Exam and interview reasoning

**E10 (defend).** *"Why not just use a comprehension for everything?"* Answer in three sentences
with a concrete threshold for when you would not.

**E11 (design).** You must compare twelve model configurations, each a dictionary of settings
derived from a base configuration. Describe how you would build the twelve without any of them
affecting the others, and name the specific bug you are guarding against.

### Transfer to a different situation

**E12 (design).** A JSON API returns a list of orders, each with a nested `customer` dictionary,
and some orders are missing the `discount` field entirely. Sketch (in five lines of pseudocode)
how you would produce a flat list of `(order_id, customer_city, discount)` tuples with a sensible
default for the missing field. Name the two Python features you rely on.

### Explain it to someone non-technical

**E13 (explain).** In under 60 words, explain to a colleague why `tuned = baseline` did not give
them a separate copy. Use an everyday comparison and say where it stops being accurate.

### Optional challenge

**E14 (code).** Write `group_by(records, key_field)` that returns a dict mapping each distinct
value of `key_field` to the list of records having it - your own `groupby`. Then use it to
recompute E6's answer, and say what pandas gives you that yours does not.

In [ ]:
# Your workspace. Still in memory: readings, summarise, record_safely, sensors, temps.

## Mastery check

Without scrolling up, can you:

- [ ] Say when a list is the right shape and when a dictionary is? *(If not: "Lists and
      dictionaries".)*
- [ ] Read a comprehension aloud in its three parts? *(If not: "Comprehensions".)*
- [ ] State the rule about mutable default arguments? *(If not: "Failure lab".)*
- [ ] Explain what `b = a` actually copies? *(If not: "The same bug, wearing different clothes".)*
- [ ] Use `key=` with `sorted`, `min` and `max` without looking it up? *(If not: "Predict before
      running".)*

## What should now feel instinctive

1. **`min`, `max` and `sorted` take a `key`** - so you get the whole record back, not just the
   number.
2. **Assignment never copies.** If you need an independent object, say so.
3. **Never a list or dict as a default argument.** `None` plus a guard.
4. **Guard clauses first**, so the main path stays unindented.
5. **`zip` truncates silently** - use `strict=True` when the lengths must match.

## Flashcards

| Question | Answer |
|---|---|
| List or dict? | List for order and iteration; dict for lookup by name |
| Three parts of a comprehension | What I want; where it comes from; which ones |
| `d["k"]` vs `d.get("k", x)` | Raise on a missing key vs return a default - choose loud or quiet deliberately |
| Why is `log=[]` dangerous as a default? | Defaults are evaluated once at definition, so every call shares one list |
| The safe pattern | `def f(x, log=None): log = [] if log is None else log` |
| What does `b = a` copy? | The reference. There is one object with two names |
| Shallow vs deep copy | Shallow copies the top level; nested lists are still shared. `copy.deepcopy` copies all the way down |
| Why not compare floats with `==`? | Binary approximation leaves tiny errors; use a tolerance |
| `/` vs `//` | True division (float) vs floor division (int) |
| What does `zip` do with unequal lengths? | Stops at the shortest, silently. `strict=True` raises instead |

## Next

**01-03 · NumPy: arrays, shapes, and vectorised thinking.**

Everything here worked on lists of records, one item at a time. That is the right tool for
reading messy input and the wrong tool for arithmetic on a million numbers. NumPy replaces the
loop with an operation on a whole array - which is faster, shorter, and introduces the single
most common numerical bug in machine learning code: getting the axis wrong.

New terms are in [GLOSSARY.md](../../GLOSSARY.md).